# SODE-Guard v2 re-run on ICS3D (Kaggle, unattended)

The same experiments as `v2_rerun_ics3d_colab.ipynb`, set up to run **in the background** with **Save Version → Save & Run All (Commit)**. You can close the browser while it runs. Each run stops itself before Kaggle's 12-hour limit, so its output is always saved. The next run picks up from that output.

### One-time setup (right-hand panel, *Settings* and *Add Input*)
1. **Accelerator: GPU T4 x2.** Not the P100: recent PyTorch builds no longer support its architecture. Not the TPU: this code is CUDA PyTorch, and the SDE solver and tree models would not run on a TPU without a rewrite.
2. **Internet: On.** Needed to clone the code. Kaggle asks for phone verification once.
3. **Add Input → Datasets →** search `integrated-cloud-security-3datasets-ics3d` and add it. The CSV is then read from `/kaggle/input`, with no API token.
4. Leave **Persistence** off. The output folder `/kaggle/working` is what carries results between runs.

### Running
* **Run 1:** click **Save Version → Save & Run All (Commit)**, then close the tab. Progress is in *View Active Events* → the run → *Logs*.
* **Run 2, 3, … (if the status table at the end says jobs are unfinished):**
  1. Open the notebook in edit mode.
  2. *Add Input → Notebook Output Files →* choose **this notebook**, select the latest version.
  3. Commit again.

  Each run copies the previous output into `/kaggle/working` and skips everything already finished.
* The core plan (12 models × 5 seeds, lite attacks, certificates) is expected to need one or two runs. Set `EXTRAS` in the next cell to add the ablations, the full-strength attack check and the per-class time split in later runs.
* GPU time counts against Kaggle's weekly GPU quota (shown on your profile).

### Results
At the end of every run, `v2_results.zip` (no model weights) and `v2/summary.md` are in the run's **Output** tab. Download the zip and upload it to the chat.

In [ ]:
import time, os
START = time.time()
HOURS = 11.0          # stop all work after this many hours; keep below Kaggle's 12-hour limit
SEEDS = '0 1 2 3 4'
EXTRAS = []           # later runs: e.g. ['ablations'], then ['fullcheck', 'classtime']
RUN_SMOKE = True      # quick check that every stage runs; skipped automatically once real results exist
OUT = '/kaggle/working/v2'
CODE = '/kaggle/temp/CV'   # code lives outside /kaggle/working so it is not saved as output
os.makedirs(OUT, exist_ok=True)
print('budget ends', time.ctime(START + HOURS * 3600))

## 1. GPU check

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'No GPU: set Accelerator to GPU T4 x2 in the right-hand panel'
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), 'compute capability', torch.cuda.get_device_capability(i))
    torch.ones(2, device=f'cuda:{i}').sum().item()      # fails here if PyTorch does not support this GPU
print('GPUs usable:', torch.cuda.device_count())

## 2. Code

In [ ]:
!rm -rf {CODE} && mkdir -p /kaggle/temp && git clone -q --depth 1 -b claude/implement-ode-guard-A39oM https://github.com/rogerpanel/CV.git {CODE}
%cd {CODE}/SODE-Guard
!git log -1 --format='code commit: %h %ci'
!python -c "import lightgbm, xgboost, sklearn, scipy" 2>/dev/null || pip -q install lightgbm xgboost scikit-learn scipy

## 3. Resume from an earlier run (if its output is attached)

In [ ]:
import glob, shutil
prev = sorted(glob.glob('/kaggle/input/**/v2/full.npz', recursive=True))
if prev:
    root = os.path.dirname(os.path.dirname(prev[-1]))
    for name in ('v2', 'v2_fullcheck', 'v2_classtime'):
        src = os.path.join(root, name)
        if os.path.isdir(src):
            shutil.copytree(src, f'/kaggle/working/{name}', dirs_exist_ok=True)
    n_done = len(glob.glob(f'{OUT}/seed_*/*.pt')) + len(glob.glob(f'{OUT}/seed_*/*.pkl'))
    print('resumed from', root, '|', n_done, 'trained models found')
else:
    print('no earlier output attached: starting fresh')

## 4. Prepare the data (skipped when resuming)

In [ ]:
if not os.path.exists(f'{OUT}/full.npz'):
    hits = glob.glob('/kaggle/input/**/Containers_Dataset.csv', recursive=True)
    assert hits, 'Add the ICS3D dataset: Add Input -> Datasets -> integrated-cloud-security-3datasets-ics3d'
    CSV = hits[0]
    print('using', CSV, f'{os.path.getsize(CSV)/1e6:.0f} MB')
    !python scripts/v2_run.py prepare --csv "{CSV}" --label-col Label --drop-cols "Src Port" --out {OUT} 2>&1 | tail -4
import json
meta = json.load(open(f'{OUT}/meta.json'))
print(json.dumps({k: meta[k] for k in ('classes', 'class_counts_after_dedup', 'dropped_small_classes', 'dedup')}, indent=1))

## 5. Smoke test (about 5 minutes, first run only)

In [ ]:
if RUN_SMOKE and not glob.glob(f'{OUT}/seed_*/results.json'):
    TEST = '/kaggle/temp/v2_smoke'
    !rm -rf {TEST} && mkdir -p {TEST} && cp {OUT}/full.npz {OUT}/meta.json {TEST}/
    !python scripts/v2_driver.py --out {TEST} --seeds 0 --deadline {time.time() + 1800} --margin 60 --heartbeat 120 --extra-args "--epochs 1 --n-val-robust 200 --n-attack 100 --restarts 1 --long-steps 5 --eot 2 --eot-apgd 2 --apgd-iters 5 --eot-steps 5 --square-queries 20 --em-paths 16 --n-cert 50 --smooth-n 200 --sanity-seeds" | tail -20
else:
    print('skipped')

## 6. Main run (time-budgeted, both GPUs)

Seeds go to the two GPUs in turn (0, 2, 4 on one and 1, 3 on the other). Each seed runs train → attack → certify, and `aggregate` runs at the end. Every 10 minutes the log shows the latest lines of each worker.

In [ ]:
deadline = START + HOURS * 3600
extras = ('--extras ' + ' '.join(EXTRAS)) if EXTRAS else ''
!python scripts/v2_driver.py --out {OUT} --seeds {SEEDS} --deadline {deadline} {extras}

## 7. Results and download

In [ ]:
import zipfile
if os.path.exists(f'{OUT}/summary.md'):
    print(open(f'{OUT}/summary.md').read())
keep = ('.md', '.json', '.log', '.npz')
with zipfile.ZipFile('/kaggle/working/v2_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root in (OUT, OUT + '_fullcheck', OUT + '_classtime'):
        for f in glob.glob(f'{root}/**/*', recursive=True):
            if f.endswith(keep) and os.path.basename(f) not in ('full.npz', 'data.npz') and '.tmp.' not in f:
                z.write(f, os.path.relpath(f, '/kaggle/working'))
print('v2_results.zip:', round(os.path.getsize('/kaggle/working/v2_results.zip') / 1e6, 1), 'MB (Output tab)')